In [2]:
import pandas as pd
import numpy as np
df_farmacs_dispensats=pd.read_csv(r'C:\Users\laiar\Desktop\DATOS_PADRIS_nets\23_antipsicotics_dispensats.csv')

C:\Users\laiar\AppData\Local\Temp\ipykernel_30972\2563718849.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_farmacs_dispensats=pd.read_csv(r'C:\Users\laiar\Desktop\DATOS_PADRIS_nets\23_antipsicotics_dispensats.csv')


In [3]:
df_farmacs_dispensats.columns

Index(['id', 'idCas', 'Any_Mes', 'ID_HIS_Subgrup_7_ATC',
       'DES_HIS_Subgrup_7_ATC', 'ID_HIS_Producte', 'DES_HIS_Producte',
       'HIS_Quantitat_PA_Espec', 'ID_HIS_Unitat_Mesura_PA_Espec',
       'DES_HIS_Unitat_Mesura_PA_Espec', 'HIS_Unitats_Productes',
       'ID_HIS_Forma_Farmaceutica', 'DES_HIS_Forma_Farmaceutica',
       'ID_HIS_Via_Administracio_Producte',
       'DES_HIS_Via_Administracio_Producte', 'Nombre_Receptes_Dispensades',
       'Nombre_DDD_Receptes_Dispensades', 'Import_Integre_Dispensades',
       'Import_Liquid_Dispensat', 'Nombre_Envasos_Dispensats'],
      dtype='object')

In [4]:
df_farmacs_dispensats['DES_HIS_Forma_Farmaceutica'].unique()

array(['COMPRIMITS', 'SOLUCIONS', 'INJECTABLES EF', 'CÃ\x80PSULES',
       'SUSPENSIONS'], dtype=object)

In [5]:
df_farmacs_dispensats['DES_HIS_Forma_Farmaceutica'].value_counts()

DES_HIS_Forma_Farmaceutica
COMPRIMITS        6171269
INJECTABLES EF    1408566
SOLUCIONS          367358
CÃPSULES          193151
SUSPENSIONS           125
Name: count, dtype: int64

### 1. Classification Oral vs. LAI products
DES_HIS_Forma_Farmaceutica we can classify:
- INJECTABLES EF → LAI
- COMPRIMITS, SOLUCIONS, CÀPSULES → ORAL
- SUSPENSIONS → REVISE (route_desc)

In [6]:
PATH_DISPENSING = r'C:\Users\laiar\Desktop\DATOS_PADRIS_nets\23_antipsicotics_dispensats.csv'
ENCODING = 'utf-8'

In [8]:
# Just to check that the relevant columns are correct
df_farmacs_dispensats[[
    "ID_HIS_Producte", "DES_HIS_Producte",
    "ID_HIS_Subgrup_7_ATC", "DES_HIS_Subgrup_7_ATC",
    "ID_HIS_Forma_Farmaceutica", "DES_HIS_Forma_Farmaceutica",
    "ID_HIS_Via_Administracio_Producte", "DES_HIS_Via_Administracio_Producte",
]]

,ID_HIS_Producte,DES_HIS_Producte,ID_HIS_Subgrup_7_ATC,DES_HIS_Subgrup_7_ATC,ID_HIS_Forma_Farmaceutica,DES_HIS_Forma_Farmaceutica,ID_HIS_Via_Administracio_Producte,DES_HIS_Via_Administracio_Producte
0,728196,ABILIFY 10MG 28 COMPRIMIDOS,N05AX12,Aripiprazol,CO,COMPRIMITS,B21,ORAL
1,728196,ABILIFY 10MG 28 COMPRIMIDOS,N05AX12,Aripiprazol,CO,COMPRIMITS,B21,ORAL
2,723799,INVEGA 3MG 28 COMPRIMIDOS DE LIBERACION PROLON...,N05AX13,Paliperidona,CO,COMPRIMITS,B21,ORAL
3,661762,QUETIAPINA STADA 100MG 60 COM RE P BLIS PVC/AI...,N05AH04,Quetiapina,CO,COMPRIMITS,B21,ORAL
4,723799,INVEGA 3MG 28 COMPRIMIDOS DE LIBERACION PROLON...,N05AX13,Paliperidona,CO,COMPRIMITS,B21,ORAL
...,...,...,...,...,...,...,...,...
8140464,700662,"XEPLION 150MG 1 JERINGA PRECARG 1,5ML SUSPEN I...",N05AX13,Paliperidona,IJ,INJECTABLES EF,B11,INTRAMUSCULAR
8140465,716976,"LATUDA 18,5MG 28 COMPRIMIDOS RECUBIERTOS CON P...",N05AE05,Lurasidona,CO,COMPRIMITS,B21,ORAL
8140466,700662,"XEPLION 150MG 1 JERINGA PRECARG 1,5ML SUSPEN I...",N05AX13,Paliperidona,IJ,INJECTABLES EF,B10,PARENTERAL
8140467,700662,"XEPLION 150MG 1 JERINGA PRECARG 1,5ML SUSPEN I...",N05AX13,Paliperidona,IJ,INJECTABLES EF,B11,INTRAMUSCULAR


In [9]:
products = df_farmacs_dispensats.drop_duplicates(subset=["ID_HIS_Producte"])

In [11]:
products= products.reset_index(drop=True)

In [13]:
# Classifiquem per tipus ORAL, LAI, o REVISAR
FORM_TO_TIPUS = {
    "COMPRIMITS":     "ORAL",
    "SOLUCIONS":      "ORAL",
    "CÀPSULES":       "ORAL",
    "CÃPSULES":       "ORAL",
    "INJECTABLES EF": "LAI",
    "SUSPENSIONS":    "ORAL",  # només GUASTIL
}

def classify_by_form(form_desc) -> str:
    if pd.isna(form_desc):
        return "REVISAR"
    norm = str(form_desc).strip().upper()
    return FORM_TO_TIPUS.get(norm, "REVISAR")

In [14]:
# For every row we classify by ORAL, LAI, or REVIEW
products['tipus'] = products['DES_HIS_Forma_Farmaceutica'].apply(classify_by_form)

In [15]:
# Generating the CSV file with the product dictionary
products.to_csv("diccionari_productes_classificats.csv", index=False, encoding="utf-8-sig")

### 2. Convert each dispensing event into days covered
For each row (=one prescription fill).

Each unique product is tagged as `ORAL` or `LAI`. We now join that back onto the full dataset so every row knows its type:

In [17]:
products[products["tipus"] == "LAI"]["DES_HIS_Producte"].unique()

array(['XEPLION 100MG 1 JERINGA PRECARG 1ML SUSPENS INYEC LIBERAC PROLONG',
       'ABILIFY MAINTENA 400MG 1 VIAL POLVO +1 VIAL DISOLV SUSPENS LIBER PROLONG',
       'ABILIFY MAINTENA 300MG 1 VIAL POLVO +1 VIAL DISOLV SUSPENS LIBER PROLONG',
       'XEPLION 75MG 1 JERINGA PRECARG 0,75ML SUSPEN INYEC LIBERAC PROLONG',
       'XEPLION 150MG 1 JERINGA PRECARG 1,5ML SUSPEN INYEC LIBERAC PROLONG',
       'RISPERDAL CONSTA 25MG/VIAL 1 VIAL + 1 JER PRECARG',
       'RISPERDAL CONSTA 37,5MG/VIAL 1 VIAL + 1 JER PRECAR',
       'TREVICTA 175MG 1 JERINGA PREC 0,875ML SUSP INYECT LIBERACION PROL',
       'XEPLION 50MG 1 JERINGA PRECARG 0,5ML SUSPEN INYEC LIBERAC PROLONG',
       'TREVICTA 350MG 1 JERINGA PRECARGADA 1,750ML SUSP INY LIBER PROLONG',
       'TREVICTA 525MG 1 JERINGA PRECARGADA 2,625ML SUSP INY LIBER PROLONG',
       'RISPERDAL CONSTA 50MG/VIAL 1 VIAL + 1 JER PRECARG',
       'XEPLION 100MG 1 JER PREC + 2 AGU SUSPENS INYECT LIBERACION PROLONGADA',
       'XEPLION 150MG 1 JER PREC + 2 

In [ ]:
# LAI interval lookup (days covered per injection)
LAI_INTERVALS = {
    # 2-weekly
    "RISPERDAL CONSTA":  14,
    "CLOPIXOL DEPOT":    14,
    "FLUANXOL DEPOT":    14,
    "HALDOL DECANOAS":   14,
    "MODECATE":          14,

    # Monthly
    "ABILIFY MAINTENA":  28,
    "XEPLION":           28,
    "INVEGA SUSTENNA":   28,
    "ZYPADHERA":         28,

    # 3-monthly
    "INVEGA TRINZA":     84,
    "TREVICTA":          84,
}

def get_lai_interval(product_desc: str):
    desc = str(product_desc).upper()
    for keyword, interval in LAI_INTERVALS.items():
        if keyword in desc:
            return interval
    return None  # unknown LAI — revisar

# Merge el tipus de producte al dataframe complet
df_farmacs_dispensats = df_farmacs_dispensats.merge(
    products[["ID_HIS_Producte", "tipus"]],
    on="ID_HIS_Producte",
    how="left"
)

# Netejar n_ddd (comma decimal → float) 
df_farmacs_dispensats["Nombre_DDD_Receptes_Dispensades"] = (
    df_farmacs_dispensats["Nombre_DDD_Receptes_Dispensades"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

# Calcular days_covered
df_farmacs_dispensats["days_covered"] = None

# Oral: agafem directament el valor de n_ddd
df_farmacs_dispensats.loc[
    df_farmacs_dispensats["tipus"] == "ORAL", "days_covered"
] = df_farmacs_dispensats["Nombre_DDD_Receptes_Dispensades"]

# LAI: apliquem el lookup d'intervals
df_farmacs_dispensats.loc[
    df_farmacs_dispensats["tipus"] == "LAI", "days_covered"
] = df_farmacs_dispensats.loc[
    df_farmacs_dispensats["tipus"] == "LAI", "DES_HIS_Producte"
].apply(get_lai_interval)

# Verificació ràpida
print(df_farmacs_dispensats[["ID_HIS_Producte", "DES_HIS_Producte", "tipus", "Nombre_DDD_Receptes_Dispensades", "days_covered"]].head(20))
print("\nNuls a days_covered:", df_farmacs_dispensats["days_covered"].isna().sum())
print("\nResum per tipus:")
print(df_farmacs_dispensats.groupby("tipus")["days_covered"].describe())